In [1]:
import requests

In [2]:
api_url = "https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api"

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def get_robust_session() -> requests.Session:
    """Tạo một HTTP Session với cơ chế tự động Retry chuẩn mực."""
    session = requests.Session()
    
    # Cấu hình retry: thử lại 3 lần, backoff factor = 1 (đợi 1s, 2s, 4s...)
    # Chỉ retry với các HTTP status codes báo lỗi từ phía server
    retries = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    
    adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    return session

In [4]:
def extract_paginated_data(api_url: str) -> list[dict]:
    session = get_robust_session()
    all_records = []
    
    # Cấu hình phân trang theo chuẩn API của bạn
    page = 1
    page_size = 1000

    print(f"Bắt đầu kéo dữ liệu từ {api_url}...")

    while True:
        # Tùy thuộc vào API, tham số truyền vào có thể là 'page' hoặc 'offset'
        # Nếu API dùng offset: offset = (page - 1) * page_size
        params = {
            "page": page,
            "limit": page_size 
        }

        try:
            response = session.get(api_url, params=params, timeout=(3.0, 10.0))
            response.raise_for_status()
            
            # Parse JSON
            data = response.json()
            
            # Trích xuất mảng chứa records. 
            # (Thay đổi key "data" thành key thực tế của API nếu JSON có dạng {"data": [...]})
            # Nếu API trả về trực tiếp một mảng [...], thì records = data
            records = data.get("data", []) if isinstance(data, dict) else data

            # --- ĐIỀU KIỆN DỪNG CỐT LÕI ---
            # Nếu mảng rỗng (độ dài = 0), thoát khỏi vòng lặp
            if not records:
                print(f"\nTrang {page} trả về rỗng. Đã quét cạn dữ liệu!")
                break

            all_records.extend(records)
            print(f"Đã kéo thành công trang {page} ({len(records)} records). Tổng: {len(all_records)}")
            
            # Tăng trang để lấy lô tiếp theo
            page += 1

        except requests.exceptions.RequestException as e:
            print(f"\nLỗi khi gọi API tại trang {page}: {e}")
            # Tùy logic pipeline, bạn có thể raise exception để báo fail task hoặc break để lấy những gì đã kéo được
            raise e 
            
    return all_records

In [6]:
import pandas as pd

In [10]:
def process_and_load(api_url: str):
    """Pipeline hoàn chỉnh: Extract -> Transform (Polars) -> Load (Postgres)."""
    # 1. Extract
    print("Bắt đầu kéo dữ liệu từ API...")
    raw_data = extract_paginated_data(api_url)
    
    if not raw_data:
        print("Không có dữ liệu để xử lý.")
        return
        
    df = pd.DataFrame(raw_data)
    
    print(f"Đã Transform xong DataFrame với kích thước: {df.shape}")
    return df


# Cách gọi:
df = process_and_load(api_url)

Bắt đầu kéo dữ liệu từ API...
Bắt đầu kéo dữ liệu từ https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api...
Đã kéo thành công trang 1 (1000 records). Tổng: 1000
Đã kéo thành công trang 2 (1000 records). Tổng: 2000
Đã kéo thành công trang 3 (1000 records). Tổng: 3000
Đã kéo thành công trang 4 (1000 records). Tổng: 4000
Đã kéo thành công trang 5 (1000 records). Tổng: 5000
Đã kéo thành công trang 6 (1000 records). Tổng: 6000
Đã kéo thành công trang 7 (1000 records). Tổng: 7000
Đã kéo thành công trang 8 (1000 records). Tổng: 8000
Đã kéo thành công trang 9 (1000 records). Tổng: 9000
Đã kéo thành công trang 10 (1000 records). Tổng: 10000

Trang 11 trả về rỗng. Đã quét cạn dữ liệu!
Đã Transform xong DataFrame với kích thước: (10000, 18)


In [11]:
df.head()

,End_Lat,End_Lon,Fare_Amt,Passenger_Count,Payment_Type,Rate_Code,Start_Lat,Start_Lon,Tip_Amt,Tolls_Amt,Total_Amt,Trip_Distance,Trip_Dropoff_DateTime,Trip_Pickup_DateTime,mta_tax,store_and_forward,surcharge,vendor_name
0,40.742963,-73.980072,45.0,1,Credit,None,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-14 23:48:00,2009-06-14 23:23:00,None,NaN,0.0,VTS
1,40.740187,-74.005698,6.5,1,Credit,None,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 17:43:00,2009-06-18 17:35:00,None,NaN,1.0,VTS
2,40.718043,-74.004745,12.5,5,Credit,None,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 18:27:00,2009-06-10 18:08:00,None,NaN,1.0,VTS
3,40.739637,-73.985233,4.9,1,CASH,None,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-14 23:58:00,2009-06-14 23:54:00,None,NaN,0.5,VTS
4,40.730032,-73.852693,25.7,1,CASH,None,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 13:23:00,2009-06-13 13:01:00,None,NaN,0.0,VTS


In [14]:
df["Trip_Dropoff_DateTime"].min()

'2009-06-01 11:48:00'

In [23]:
proportion = len(df[df["Payment_Type"] == "Credit"]) / len(df)
proportion

0.2666

In [25]:
df["Tip_Amt"].sum()

np.float64(6063.41)